# Assignment: Conservative NN

Conventional neural networks are pure pattern matchers: they fit whatever
function minimizes the training loss, with no notion of physical plausibility.
In many scientific and engineering domains, however, we know part of the
truth in advance (a conservation law, a symmetry, a monotonicity constraint)
and we would like the model to respect it *by construction*, not just
approximately after enough training data.

> For example, consider a river basin instrumented with several flow gauges. At each time step, the water flow measured at a downstream station should equal the combined upstream inflows, plus local rainfall contributions, minus losses due to evaporation, infiltration, or water extraction. A conventional neural network might predict the downstream flow directly from historical measurements and weather variables. Although its predictions may fit the observed data well, nothing prevents the model from implicitly creating or removing water, producing estimates that violate the conservation of mass. A conservative neural network, instead, predicts only the unknown local gains and losses, while the downstream flow is computed using the mass balance equation. As a result, every prediction automatically satisfies the conservation law while still allowing the model to learn the complex hydrological processes that cannot be described analytically.

In this assignment you will build a small but complete example of this idea:
a **water reservoir** whose mass balance is governed by a known conservation
law. You will train two recurrent models on the same data and the same loss
function:

1. A **conventional LSTM** (baseline, Option 1) that predicts the next mass value directly.
2. **Your chosen architecture** (Option 2, 3, or 4 from §2) that incorporates the conservation law.

Both are functions of the same inputs, trained with the same MSE loss and the
same optimizer. The only difference is architectural: one has the
conservation law baked into its output layer, the other does not. Comparing
them lets you measure, empirically, what a **physical constraint** buys
you over a network that has to *learn* the constraint from data alone.

The implementation will be carried out in **JAX** (with **Flax** for the
neural network layers and **Optax** for optimization).

### Learning objectives

By the end of this assignment you should be able to:

- Explain different ways to use physical constraints in neural networks.
- Implement, train, and debug a recurrent model in JAX/Flax.
- Design an architecture that provably satisfies a known physical law.
- Quantitatively evaluate model quality along more than one axis (accuracy
  *and* physical consistency).
- Reason about physical knowledge as a form of **inductive bias**, and its
  effect on data efficiency.

## 1. Assignment Structure

This assignment runs in two phases.

### Phase A — In-class group exercise (no code)

**Goal**: understand the problem *before* writing any code.

In small groups, discuss and produce a short written plan (a few bullet
points per item is enough) covering:

1. **Design a specific scenario.** Draw the reservoir system you will consider: what flows in, what flows out, what is measured, what is hidden. Write down the conservation
   equation in your own words.
2. **Why a plain LSTM can fail.** Give a concrete example of a prediction a
   conventional LSTM could make that would be physically impossible in this
   system.
3. **The solution space.** There are different ways you could make a
   neural network respect the conservation law (see below). For each alternative, discuss:
    1. Which approaches guarantee conservation?
    2. Which are easiest to implement?
    3. Which are likely to generalize best with limited data?
    4. Which would you choose for this scenario? Justify your answer.
4. **A plan.** Sketch, at a high level, the architecture you will implement
   at home, and the metrics you will use to judge whether it worked.

Bring this plan to the individual phase — you will implement it.

### Phase B — Individual take-home implementation

Goal: implement, train, evaluate, and reflect. See
`A1_Phase_B_Individual_Exercise.ipynb` for the full task breakdown and
deliverables checklist.


# Phase A — Classroom Group Exercise

## Conservative NN architectures: Understanding the Problem

**Format:** small groups, in class, **no coding**.

**Goal:** understand the physical scenario and the design problem *before* writing any code. What your group produces here is the plan you will implement individually in Phase B (`A1_Phase_B_Individual_Exercise.ipynb`).

Fill in your group's answers directly in the empty markdown cells below (double-click a cell to edit it, then run it to render).

## Objectives of this session

- Understand why a conventional neural network can produce physically impossible predictions.
- Explore the water-reservoir scenario and its conservation law.
- Compare candidate solution strategies for enforcing the law.
- Produce a short written plan — architecture + metrics — for the individual implementation in Phase B.

## 1. The scenario: a water reservoir

We model a water reservoir observed over discrete time steps $t = 0, 1, 2, ...$. At each step:

| Symbol | Meaning | Observed? |
|---|---|---|
| $m(t)$ | Water mass (volume) in the reservoir | yes |
| $r(t)$ | Inflow (e.g. rainfall, pumping in) | yes |
| $q(t)$ | Controlled outflow (e.g. a valve setting) | yes |
| $c(t)$ | Hidden consumption (e.g. evaporation, leakage, unmetered usage) | **no** |

The system evolves according to an exact mass balance:

$$
m(t+1) = m(t) + r(t) - q(t) - c(t)
$$

$c(t)$ is *not* directly observable. It depends on temperature, time of day, day of the year and a small random component, and must be inferred from context. It is exactly the term a model would have to estimate.

## 2. Physical knowledge and inductive bias

There is more than one way to incorporate the conservation law into a neural network. Each way implement a different **inductive bias** in the network. 

Discuss the following alternatives and decide which one is most appropriate for this problem. For each option, consider its advantages and disadvantages in terms of **physical consistency, ease of training, interpretability, and generalization**.

### Option 1. Direct Prediction (Baseline - no conservation law)

The network (a conventional LSTM) directly predicts the next water mass:

$$\hat m(t+1)=\mathrm{LSTM}(x(t))$$

At each time step, the input vector $x(t)$ contains the current water mass $m(t)$, the inflow $r(t)$, the controlled outflow $q(t)$, and any other available variables such as temperature , time of day, and day of the year. However, the system is also influenced by hidden factors that are not directly measured, including evaporation, soil infiltration, small leaks, unreported water withdrawals, and other environmental effects. These hidden processes determine an unknown water loss $c(t)$, which cannot be observed directly but must be inferred from the available measurements.

No conservation law is enforced. The model is free to learn any mapping that minimizes the prediction error.

Advantages

* Simplest architecture.
* Maximum flexibility.

Disadvantages

* Predictions may violate the conservation law.
* Requires the network to learn physical principles from data alone.

### Option 2. Conservation as a Soft Constraint

The network has two outputs: it predicts both the next water mass and the estimated water consumption,

$$(\hat m(t+1),\,\hat c(t)) = \mathrm{LSTM}(x(t)).$$

The prediction loss is combined with an additional penalty that measures how well the predicted quantities satisfy the conservation law:

$$L = L_{\text{prediction}} + \lambda L_{\text{physics}},$$

where

$$L_{\text{prediction}}
=
\big(\hat m(t+1)-m(t+1)\big)^2,$$

and

$$L_{\text{physics}}
=
\left(
\hat m(t+1)-m(t)-r(t)+q(t)+\hat c(t)
\right)^2.
$$

The parameter $\lambda$ controls the trade-off between prediction accuracy and adherence to the conservation law.

Advantages

* Easy to incorporate into an existing neural network.
* Encourages the model to learn physically consistent predictions.
* Allows some flexibility when the governing equation is only approximately valid or when measurements are noisy.

Disadvantages

* Conservation is only encouraged, not guaranteed.
* The quality of the solution depends on the choice of the penalty weight \lambda.
* The model must learn two related quantities simultaneously, which may make training more difficult and slower to converge.

### Option 3. Conservative Architecture (Hard Constraint)

The network predicts only the unknown hidden consumption:

$$\hat c(t)=\mathrm{LSTM}(x(t))$$

The final water mass is computed analytically:

$$\hat m(t+1)=m(t)+r(t)-q(t)-\hat c(t).$$

Advantages

* Conservation is satisfied exactly.
* The network focuses only on learning the unknown process.
* Strong inductive bias can improve generalization.

Disadvantages

* Requires that the governing equation is known.
* Less flexible if the physical model is approximate.
  
### Option 4. Predict Residual Corrections

Suppose the conservation equation is only approximately correct. The network predicts a small correction:

$$\hat m(t+1) = m(t)+r(t)-q(t)-\hat c(t) +\delta(t), $$

where the network learns both the hidden consumption and the correction term.

Advantages

* Can compensate for modeling errors.
* Useful when the physics is only approximate.

Disadvantages

* The correction may absorb physical inconsistencies.
* Conservation is no longer exact.

The goal is not simply to obtain the lowest prediction error, but to understand how prior scientific knowledge can be integrated into neural network design, and how different design choices trade off flexibility, interpretability, and physical consistency. Discuss these four options as a group in Discussion 3 below.

## Discussion 1 — Draw the system

As a group, sketch the reservoir system: what flows in, what flows out, what is measured, what is hidden. Write the conservation equation in your own words (not just the symbols — explain what it means physically).

You can draw an ASCII/text diagram here, or describe it in words.

_Group answer — Discussion 1:_

**The system.** One reservoir, observed once per hour.

```
        rainfall / pumping in                     evaporation, leakage,
              r(t)  [measured]                    unmetered use  c(t)  [HIDDEN, >= 0]
                 │                                          ▲
                 ▼                                          │
        ┌──────────────────────────────────────────────────┴──┐
        │              RESERVOIR   m(t)  [measured]            │
        └──────────────────────────┬──────────────────────────┘
                                   ▼
                        valve outflow  q(t)  [measured]

   context sensors (measured): temperature, hour of day, day of year
```

- **Measured:** the stored volume `m(t)`, the inflow `r(t)`, the valve outflow `q(t)`, and three context variables (temperature, hour, day of year).
- **Hidden:** the consumption `c(t)`. Nobody meters it. It rises with temperature (evaporation) and follows a daily usage profile, with a small random part. It can never be negative: evaporation and leaks only remove water.

**The conservation equation, in words.** The water in the reservoir one hour from now is the water there now, plus what came in, minus what was released through the valve, minus what was lost in ways we do not measure:

$$m(t+1) = m(t) + r(t) - q(t) - c(t), \qquad c(t) \ge 0$$

Water is neither created nor destroyed inside the reservoir. Three of the four terms on the right are measured, so `c(t)` is the only unknown, and it is fixed by the others: `c(t) = m(t) + r(t) - q(t) - m(t+1)`. Predicting `m(t+1)` and estimating `c(t)` are therefore the same problem.

## Discussion 2 — Why can a plain LSTM fail?

Give a **concrete example** of a prediction a conventional LSTM (trained to predict `m(t+1)` directly from `[m(t), r(t), q(t), temp(t), hour(t), day_year(t)]`, with an MSE loss) could plausibly make that would be **physically impossible** in this system.

_Group answer — Discussion 2:_

**Example.** At some hour the reservoir holds `m(t) = 150.0`, the inflow is `r(t) = 3.0` and the valve releases `q(t) = 1.5`. Without any loss the next volume would be `150.0 + 3.0 - 1.5 = 151.5`. Since losses are never negative, every physically possible value satisfies `m(t+1) <= 151.5`.

Suppose the true consumption was `c(t) = 0.4` (a cool night), so the true value is `m(t+1) = 151.1`, and the LSTM predicts `151.9`.

- As a prediction this looks excellent: the error is 0.8 on a volume of about 150, a squared error of 0.64.
- Physically it is impossible: it implies `c(t) = 151.5 - 151.9 = -0.4`, that is, 0.4 units of water appeared from nowhere.

**Why the network does this.** The MSE loss only measures the distance to the target. A prediction of `150.3` (implied `c = +1.2`, possible) and one of `151.9` (implied `c = -0.4`, impossible) are both 0.8 away from the truth and receive exactly the same loss, so training never pushes the model away from the impossible one. The boundary `m(t+1) <= m(t) + r(t) - q(t)` is not written anywhere in the architecture or in the loss. The LSTM would have to discover it from data, and it only approximates it. The risk is highest when `c(t)` is small, because then the true value sits close to the boundary and a small error crosses it.

More data or longer training reduces how often this happens, but nothing makes it impossible.

## Discussion 3 — Choose a solution strategy

Using the four options from §2, discuss as a group:

1. Which approaches guarantee conservation?
2. Which are easiest to implement?
3. Which are likely to generalize best with limited data?
4. Which would you choose for this scenario? Justify your answer.

_Group answer — Discussion 3:_

**1. Which approaches guarantee conservation?**

| Option | Guarantee | Why |
|---|---|---|
| 1. Direct | None | The law appears nowhere. |
| 2. Soft constraint | None | The penalty encourages consistency between two free outputs; it is traded against the prediction loss through `λ`, and nothing forces `ĉ >= 0`. |
| 3. Hard constraint | **Exact** | `m̂` is computed from the balance and `softplus` keeps `ĉ >= 0`. True before training and on unseen data. |
| 4. Residual correction | Approximate | The balance is the backbone of the prediction, but `δ(t)` is unconstrained. The violation is bounded by the size of `δ`, so it can be measured and monitored. |

**2. Which are easiest to implement?** Option 1 is the simplest. Options 3 and 4 add one line of arithmetic after the network (and a second output head for Option 4) and keep the same MSE loss. Option 2 is the most work: two heads, a second loss term, and `λ` to tune with several training runs.

**3. Which are likely to generalize best with limited data?** Options 3 and 4. In both, the network no longer has to learn the large, easy part of the problem (`m + r - q`, values around 150): it only learns the small hidden term (values around 1–2) from temperature and time. That is a strong inductive bias, so fewer sequences should be needed. Option 3 has the stronger bias; Option 4 is slightly more flexible and should be close to it as long as `δ` stays small. Option 2 still has to learn the mass directly, so we expect it to behave more like the baseline when data is scarce.

**4. Our choice: Option 4 (residual correction).**

- In this exercise the simulated data obeys the balance exactly, so Option 3 would be sufficient. We choose Option 4 because a real reservoir never does: level sensors have error, and there are flows nobody meters (seepage in, runoff from the banks, rain falling directly on the surface). When the equation is incomplete, a hard constraint forces every mismatch into `ĉ`, and `ĉ >= 0` cannot represent a net unmetered gain.
- Option 4 keeps the physics as the main structure of the prediction and adds one explicit, inspectable term for whatever the equation misses. The split is interpretable: `ĉ` is the modelled loss, `δ` is the part the physics could not explain.
- It turns an assumption into a measurement. If the trained `δ(t)` is close to zero, the data confirms that the balance is complete and Option 3 would have been enough. If it is not, we learn how large the missing term is.
- It costs almost nothing: same loss, same optimizer, no new hyperparameter.

**The price, which we accept and will measure.** Conservation is no longer guaranteed, because `δ` has no positivity constraint. There is also an identifiability problem: only the difference `ĉ(t) - δ(t)` enters the prediction, so the loss cannot tell a larger `ĉ` from a more negative `δ`. We therefore expect to judge the model on the *effective* consumption `ĉ - δ` and to report the size of `δ` next to the physical error.

## Discussion 4 — Your plan

Sketch, at a high level, the architecture your will (individualy) implement at home (Phase B), and the metrics you will use to judge whether it worked. This becomes your common plan for the individual implementation. Bring it with you.

Consider:
- What does the network predict, and what (if anything) is computed analytically afterward?
- What inputs does each part of the model need — and critically, should the true hidden consumption `c(t)` ever be a model input? Why or why not?
- How will you measure success? (Think about more than one axis — e.g. prediction accuracy *and* physical consistency.)

_Group answer — Discussion 4 (your plan):_

**Architecture (Option 4).**

```
x(t), normalized ─► LSTM (hidden 32) ─► h(t) ─┬─► Dense(1) ─► softplus ─► ĉ(t) >= 0
                                              └─► Dense(1) ─────────────► δ(t)   (no constraint)

m̂(t+1) = m(t) + r(t) - q(t) - ĉ(t) + δ(t)        ◄── computed analytically, in RAW physical units
```

- **Predicted by the network:** the hidden consumption `ĉ(t)` and a correction `δ(t)`.
- **Computed analytically afterwards:** the next mass, from the balance.
- **Baseline for comparison (Option 1):** the same LSTM followed by one `Dense(1)` that outputs `m̂(t+1)` directly.

**Inputs.**

- Network input for both models: `x(t) = [m, r, q, temp, hour_norm, day_year_norm]`, normalized with training-set statistics.
- The balance uses the raw, unnormalized `m(t), r(t), q(t)`. Adding and subtracting normalized values has no physical meaning.
- **The true `c(t)` is never an input and never a training target.** It is determined by the inputs and the target (`c = m + r - q - m(t+1)`), so giving it to a model would hand over the answer. It would also be unavailable in a real deployment, where it is by definition unmeasured. We keep it only to check, after training, how well the hidden process was recovered.

**Training.** Both models with the same MSE loss on `m(t+1)`, the same optimizer (Adam, learning rate 1e-2), hidden size, epochs (300) and batch size (32 sequences), so the only difference is the architecture. Train/test split by whole sequences; normalization statistics from the training split only.

**How we will measure success (all on the held-out test sequences).**

| Axis | Metric | Question |
|---|---|---|
| Accuracy | MSE and MAE of `m̂(t+1)` | Is it right? |
| Physical consistency | Physical error `mean(max(0, -(m + r - q - m̂)))`, and the share of time steps where the implied consumption is negative | Is it possible? |
| Cost | Training time | Is the structure expensive? |
| Specific to Option 4 | Mean and maximum of `|δ(t)|`; error of `ĉ` and of `ĉ - δ` against the true `c(t)` | How much did the correction do, and was the hidden process recovered? |

**What we expect.** A much lower MSE than the baseline, because the network only learns the small unknown term. A physical error that is not guaranteed to be zero but stays near zero while `δ` is small. A baseline that implies negative consumption on some time steps. If time allows, the low-data ablation (test MSE against the number of training sequences) to show the inductive bias directly.

---

## Handoff to Phase B

Open `A1_Phase_B_Individual_Exercise.ipynb` and work through it using the plan you just wrote: you'll build a **conventional LSTM** as your baseline and **your chosen architecture** (Option 2, 3, or 4 — whichever you sketched in Discussion 4), then compare them experimentally.